# Lesson 02 - Spoznajmo Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** je enoten okvir za ustvarjanje AI agentov. Ponuja čisto, sestavljivo arhitekturo s štirimi osnovnimi gradniki:

- **Client** – poveže se z AI model endpointom in upravlja komunikacijo
- **Agent** – obdaja klienta z navodili in definicijami orodij
- **Tools** – razširijo zmožnosti agenta s prilagojenimi funkcijami, ki jih model lahko kliče
- **Session** – vzdržuje zgodovino pogovora za večkratne interakcije

V tej lekciji bomo z uporabo teh konceptov ustvarili **agenta za rezervacijo potovanj**, ki preverja razpoložljivost destinacij.


## Namestitev


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Razumevanje arhitekture Agentovega okvira

Microsoft Agent Framework sledi plastni arhitekturi:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Odjemalec** – `AzureAIProjectAgentProvider` se poveže z nameščanjem Azure OpenAI. Upravlja avtentikacijo, oblikovanje zahtevkov in razčlenjevanje odgovorov.
2. **Agent** – Ustvari se iz odjemalca prek `provider.create_agent()`. Agent združuje dostop do modela z navodili (sistemski poziv) in orodji.
3. **Orodja** – Python funkcije okrašene z `@tool`, ki jih agent lahko kliče za izvajanje dejanj ali pridobivanje podatkov.
4. **Seja** – Objekt `AgentSession` (ustvarjen prek `agent.create_session()`), ki shranjuje zgodovino pogovora in omogoča večkrožno dialog, kjer se agent spominja prejšnjega konteksta.

Zgradimo vsako plast korak za korakom.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dodajanje orodij z dekoratorjem @tool

Orodja agentom omogočajo izvajanje dejanj, ki presegajo generiranje besedila. Dekorator `@tool` pretvori običajno Python funkcijo v nekaj, kar lahko agent pokliče.

Ključne točke:
- Uporabite `Annotated[type, "description"]`, da model razume vsak parameter.
- Dokumentacijski niz postane opis orodja, ki ga model vidi.
- `approval_mode="never_require"` pomeni, da se orodje samodejno izvaja brez potrditve uporabnika.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Ustvarjanje agenta z orodji

Zdaj združimo odjemalca, navodila in orodja v agenta. `instructions` delujejo kot sistemski poziv — določajo osebnost in vedenje agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Pogovori z več poteki z uporabo sej

`AgentSession` (ustvarjena prek `agent.create_session()`) spremlja vsa sporočila v pogovoru. S tem, ko isto sejo predamo v vsak klic `agent.run()`, ima agent dostop do celotne zgodovine pogovora in se lahko sklicuje na prejšnja sporočila.

Podamo `tools=[check_destination_availability]`, da lahko agent med vsakim potekom pokliče našega preverjevalca razpoložljivosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Povzetek

V tej lekciji ste raziskali štiri stebre Microsoft Agent Frameworka:

| Koncept | Kaj ste se naučili |
|---------|--------------------|
| **Stranka** | `AzureAIProjectAgentProvider` se poveže z Azure OpenAI z avtentikacijo na podlagi poverilnic |
| **Agent** | `provider.create_agent()` združi povezavo z modelom z navodili in imenom |
| **Orodja** | Dekorator `@tool` razkrije Python funkcije, ki jih agent lahko kliče |
| **Seja** | `agent.create_session()` ohranja zgodovino pogovora skozi več krogov |

Ti gradniki skupaj sestavljajo agente, ki lahko vodijo naravne pogovore, kličejo zunanje funkcije in ohranjajo kontekst — temelj za bolj napredne agentne vzorce v poznejših lekcijah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo storitve za strojno prevajanje AI [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas opozarjamo, da lahko avtomatizirani prevodi vsebujejo napake ali nenatančnosti. Izvirni dokument v njegovem izvirnem jeziku velja za avtoritativni vir. Za ključne informacije priporočamo strokovni človeški prevod. Za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda, ne prevzemamo odgovornosti.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
